# Binary PSO for Feature Selection

**Module 06 — Evolutionary & Swarm Methods**

Estimated time: 15 minutes

## What you will build

1. Binary PSO from scratch with sigmoid transfer function and velocity clamping
2. Particle trajectory visualisation — feature selection probability over iterations
3. Convergence comparison: PSO vs GA
4. V-shaped vs S-shaped transfer function comparison

**Reference**: Kennedy & Eberhart (1995). Particle swarm optimization. *Proceedings of ICNN*, 4, 1942–1948.

In [ ]:
import warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

# ── Dataset ────────────────────────────────────────────────────────────────────
data = load_breast_cancer()
X_raw, y = data.data, data.target
X = StandardScaler().fit_transform(X_raw)
N_FEATURES = X.shape[1]
FEATURE_NAMES = data.feature_names

# 5-fold CV object (reused throughout)
CV5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f"Dataset: {X.shape[0]} samples, {N_FEATURES} features")

# Baseline
clf_base = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
baseline_scores = cross_val_score(clf_base, X, y, cv=CV5, scoring='accuracy')
BASELINE_ACC = baseline_scores.mean()
BASELINE_ERR = 1.0 - BASELINE_ACC
print(f"Baseline (all {N_FEATURES} features): {BASELINE_ACC:.4f} accuracy")

## 1. Transfer Functions

Binary PSO requires mapping continuous velocity to a bit-flip probability.

In [ ]:
def sigmoid(v: np.ndarray) -> np.ndarray:
    """S-shaped transfer function (Kennedy & Eberhart 1995)."""
    return 1.0 / (1.0 + np.exp(-np.clip(v, -500, 500)))


def v_shaped(v: np.ndarray) -> np.ndarray:
    """V-shaped transfer function (Mirjalili & Lewis 2013)."""
    return np.abs(np.tanh(v))


# Visualise both transfer functions
v_range = np.linspace(-6, 6, 300)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.plot(v_range, sigmoid(v_range), 'b-', linewidth=2.5, label='S-shaped: sigmoid(v)')
ax.plot(v_range, v_shaped(v_range), 'r--', linewidth=2.5, label='V-shaped: |tanh(v)|')
ax.axhline(0.5, color='gray', linestyle=':', alpha=0.7, label='P=0.5')
ax.axvline(-4, color='green', linestyle=':', alpha=0.7, label='Clamp boundary (±4)')
ax.axvline(4, color='green', linestyle=':', alpha=0.7)
ax.set_xlabel('Velocity v', fontsize=12)
ax.set_ylabel('P(bit = 1)', fontsize=12)
ax.set_title('Transfer Functions for Binary PSO', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim(-0.05, 1.05)

ax2 = axes[1]
# Show what happens with clamping
for v_max, color in [(2.0, 'blue'), (4.0, 'red'), (6.0, 'green')]:
    v_clamped = np.clip(v_range, -v_max, v_max)
    probs = sigmoid(v_clamped)
    ax2.plot(v_range, probs, color=color, linewidth=2, label=f'v_max={v_max}')

ax2.set_xlabel('Raw velocity v', fontsize=12)
ax2.set_ylabel('P(bit = 1) after clamping', fontsize=12)
ax2.set_title('Effect of Velocity Clamping on S-shaped', fontsize=12)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)
ax2.set_ylim(-0.05, 1.05)

plt.tight_layout()
plt.savefig('pso_transfer_functions.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"S-shaped: S(±4) = {sigmoid(np.array([4.0, -4.0]))}")  
print(f"V-shaped: V(±4) = {v_shaped(np.array([4.0, -4.0]))}")

## 2. Binary PSO Implementation

Full binary PSO with:
- Linear inertia decay
- Velocity clamping
- Configurable transfer function
- Particle trajectory recording

In [ ]:
def evaluate_mask(mask: np.ndarray, n_trees: int = 50) -> float:
    """Cross-validation error rate for a binary feature mask."""
    if not mask.any():
        return 1.0
    clf = RandomForestClassifier(n_estimators=n_trees, random_state=42, n_jobs=-1)
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    return 1.0 - cross_val_score(clf, X[:, mask], y, cv=cv, scoring='accuracy').mean()


def binary_pso(
    n_particles: int = 40,
    n_iterations: int = 50,
    w_max: float = 0.9,
    w_min: float = 0.4,
    c1: float = 2.0,
    c2: float = 2.0,
    v_max: float = 4.0,
    transfer_fn: str = 'sigmoid',
    random_state: int = 42,
    record_trajectories: bool = False,
):
    """
    Binary Particle Swarm Optimisation for feature selection.

    Parameters
    ----------
    transfer_fn : 'sigmoid' or 'vshaped'
    record_trajectories : if True, record per-particle probability vectors.

    Returns
    -------
    dict with keys:
        best_mask, best_fitness, fitness_history, trajectories (optional)
    """
    rng = np.random.default_rng(random_state)

    transfer = sigmoid if transfer_fn == 'sigmoid' else v_shaped

    # ── Initialise ────────────────────────────────────────────────────────────
    positions = rng.integers(0, 2, size=(n_particles, N_FEATURES)).astype(float)
    velocities = rng.uniform(-v_max, v_max, size=(n_particles, N_FEATURES))

    fitness = np.array([evaluate_mask(pos.astype(bool)) for pos in positions])

    personal_best_pos = positions.copy()
    personal_best_fit = fitness.copy()

    g_best_idx = np.argmin(fitness)
    global_best_pos = positions[g_best_idx].copy()
    global_best_fit = fitness[g_best_idx]

    fitness_history = [global_best_fit]
    feature_count_history = [global_best_pos.sum()]
    trajectories = [] if record_trajectories else None

    # ── Main Loop ─────────────────────────────────────────────────────────────
    for t in range(n_iterations):
        # Linearly decaying inertia
        w = w_max - (w_max - w_min) * t / n_iterations

        r1 = rng.random(size=(n_particles, N_FEATURES))
        r2 = rng.random(size=(n_particles, N_FEATURES))

        # Velocity update
        velocities = (
            w * velocities
            + c1 * r1 * (personal_best_pos - positions)
            + c2 * r2 * (global_best_pos - positions)
        )
        # Velocity clamping (prevents sigmoid saturation)
        velocities = np.clip(velocities, -v_max, v_max)

        # Binary position update via transfer function
        probs = transfer(velocities)

        if transfer_fn == 'vshaped':
            # V-shaped: flip with probability V(v), regardless of current bit
            flip_mask = rng.random(size=(n_particles, N_FEATURES)) < probs
            positions = np.abs(positions - flip_mask.astype(float))
        else:
            # S-shaped: set bit to 1 with probability S(v)
            positions = (rng.random(size=(n_particles, N_FEATURES)) < probs).astype(float)

        # Evaluate
        fitness = np.array([evaluate_mask(pos.astype(bool)) for pos in positions])

        # Update personal bests
        improved = fitness < personal_best_fit
        personal_best_pos[improved] = positions[improved].copy()
        personal_best_fit[improved] = fitness[improved]

        # Update global best
        g_idx = np.argmin(personal_best_fit)
        if personal_best_fit[g_idx] < global_best_fit:
            global_best_pos = personal_best_pos[g_idx].copy()
            global_best_fit = personal_best_fit[g_idx]

        fitness_history.append(global_best_fit)
        feature_count_history.append(int(global_best_pos.sum()))

        if record_trajectories:
            # Record sigmoid probability for each particle on each feature
            trajectories.append(sigmoid(velocities).copy())

    best_mask = global_best_pos.astype(bool)
    return {
        'best_mask': best_mask,
        'best_fitness': global_best_fit,
        'fitness_history': fitness_history,
        'feature_count_history': feature_count_history,
        'trajectories': trajectories,
    }


print("Binary PSO implementation ready.")
print("Parameters: w decays 0.9→0.4, c1=c2=2.0, v_max=4.0")

## 3. Run Binary PSO

In [ ]:
print("Running Binary PSO with S-shaped transfer (40 particles, 50 iterations)...")
pso_sigmoid = binary_pso(
    n_particles=40,
    n_iterations=50,
    transfer_fn='sigmoid',
    record_trajectories=True,
    random_state=42,
)

print(f"\nPSO (sigmoid) result:")
print(f"  Best error: {pso_sigmoid['best_fitness']:.4f}")
print(f"  Features: {pso_sigmoid['best_mask'].sum()}/{N_FEATURES}")

# Validate with 5-fold CV
clf_pso = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
pso_scores = cross_val_score(clf_pso, X[:, pso_sigmoid['best_mask']], y,
                              cv=CV5, scoring='accuracy')
print(f"  5-fold CV accuracy: {pso_scores.mean():.4f} ± {pso_scores.std():.4f}")

In [ ]:
print("Running Binary PSO with V-shaped transfer...")
pso_vshaped = binary_pso(
    n_particles=40,
    n_iterations=50,
    transfer_fn='vshaped',
    record_trajectories=False,
    random_state=42,
)

print(f"\nPSO (V-shaped) result:")
print(f"  Best error: {pso_vshaped['best_fitness']:.4f}")
print(f"  Features: {pso_vshaped['best_mask'].sum()}/{N_FEATURES}")

clf_pso_v = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
pso_v_scores = cross_val_score(clf_pso_v, X[:, pso_vshaped['best_mask']], y,
                                cv=CV5, scoring='accuracy')
print(f"  5-fold CV accuracy: {pso_v_scores.mean():.4f} ± {pso_v_scores.std():.4f}")

## 4. Particle Trajectory Visualisation

We visualise the selection probability for each feature across iterations for 5 selected particles.
A dark colour means the particle strongly prefers selecting that feature.

In [ ]:
trajectories = pso_sigmoid['trajectories']  # list of (n_particles, N_FEATURES) arrays
n_iters = len(trajectories)

# Stack into (iterations, particles, features)
traj_array = np.stack(trajectories, axis=0)  # (n_iters, n_particles, N_FEATURES)

# Show probability of selecting each feature for particle 0 over time
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Top: Heatmap of selection probabilities for particle 0
ax = axes[0]
particle_idx = 0
im = ax.imshow(
    traj_array[:, particle_idx, :].T,  # (features, iterations)
    aspect='auto',
    cmap='Blues',
    vmin=0, vmax=1,
)
plt.colorbar(im, ax=ax, label='P(feature selected)')
ax.set_xlabel('Iteration', fontsize=12)
ax.set_ylabel('Feature index', fontsize=12)
ax.set_title(f'Particle 0: Feature Selection Probabilities Over Time', fontsize=12)

# Bottom: Mean selection probability across all particles for each feature
ax2 = axes[1]
mean_probs = traj_array.mean(axis=1)  # (n_iters, N_FEATURES)

# Show 3 specific features: one that gets selected, one that doesn't, one that varies
best_mask = pso_sigmoid['best_mask']
selected_feats = np.where(best_mask)[0][:3] if best_mask.sum() >= 3 else np.where(best_mask)[0]
rejected_feats = np.where(~best_mask)[0][:2]

for f_idx in selected_feats:
    ax2.plot(mean_probs[:, f_idx], linewidth=1.5,
             label=f'{FEATURE_NAMES[f_idx]} (selected)', alpha=0.8)

for f_idx in rejected_feats:
    ax2.plot(mean_probs[:, f_idx], linewidth=1.5, linestyle='--',
             label=f'{FEATURE_NAMES[f_idx]} (rejected)', alpha=0.6)

ax2.axhline(0.5, color='gray', linestyle=':', alpha=0.7, label='Decision boundary')
ax2.set_xlabel('Iteration', fontsize=12)
ax2.set_ylabel('Mean P(feature selected) across swarm', fontsize=12)
ax2.set_title('Swarm Convergence: Feature Selection Probabilities', fontsize=12)
ax2.legend(fontsize=8, loc='upper right')
ax2.grid(True, alpha=0.3)
ax2.set_ylim(-0.05, 1.05)

plt.tight_layout()
plt.savefig('pso_trajectories.png', dpi=120, bbox_inches='tight')
plt.show()
print("Figure saved: pso_trajectories.png")

## 5. Simple GA for Comparison

In [ ]:
import random as py_random

def run_ga(
    pop_size: int = 40,
    n_generations: int = 50,
    random_state: int = 42,
) -> dict:
    """Single-objective elitist GA for direct comparison with PSO."""
    rng = py_random.Random(random_state)
    np_rng = np.random.default_rng(random_state)

    def fitness_fn(individual):
        mask = np.array(individual, dtype=bool)
        return evaluate_mask(mask)

    # Initialise
    population = [np_rng.integers(0, 2, N_FEATURES).tolist() for _ in range(pop_size)]
    fit = [fitness_fn(ind) for ind in population]

    best_fit = min(fit)
    best_ind = population[np.argmin(fit)].copy()
    fitness_history = [best_fit]
    feature_count_history = [sum(best_ind)]

    for gen in range(n_generations):
        new_pop = []
        for _ in range(pop_size):
            # Tournament selection (k=3)
            p1 = min(rng.sample(range(pop_size), 3), key=lambda i: fit[i])
            p2 = min(rng.sample(range(pop_size), 3), key=lambda i: fit[i])
            # Single-point crossover
            cx = rng.randint(1, N_FEATURES - 1)
            child = population[p1][:cx] + population[p2][cx:]
            # Bit-flip mutation
            child = [1 - b if rng.random() < 1.0 / N_FEATURES else b for b in child]
            if sum(child) == 0:
                child[rng.randint(0, N_FEATURES - 1)] = 1
            new_pop.append(child)

        new_fit = [fitness_fn(ind) for ind in new_pop]

        # Elitist: keep best N from combined
        all_pop = population + new_pop
        all_fit = fit + new_fit
        top = np.argsort(all_fit)[:pop_size]
        population = [all_pop[i] for i in top]
        fit = [all_fit[i] for i in top]

        if fit[0] < best_fit:
            best_fit = fit[0]
            best_ind = population[0].copy()

        fitness_history.append(best_fit)
        feature_count_history.append(sum(best_ind))

    return {
        'best_mask': np.array(best_ind, dtype=bool),
        'best_fitness': best_fit,
        'fitness_history': fitness_history,
        'feature_count_history': feature_count_history,
    }


print("Running GA (40 pop, 50 generations)...")
ga_result = run_ga(pop_size=40, n_generations=50, random_state=42)

clf_ga = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
ga_scores = cross_val_score(clf_ga, X[:, ga_result['best_mask']], y,
                             cv=CV5, scoring='accuracy')
print(f"GA result:")
print(f"  Best error: {ga_result['best_fitness']:.4f}")
print(f"  Features: {ga_result['best_mask'].sum()}/{N_FEATURES}")
print(f"  5-fold CV accuracy: {ga_scores.mean():.4f} ± {ga_scores.std():.4f}")

## 6. Convergence Comparison: PSO vs GA

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Left: Fitness convergence ──────────────────────────────────────────────────
ax = axes[0]

ax.plot(pso_sigmoid['fitness_history'], 'b-', linewidth=2, label='PSO (sigmoid)')
ax.plot(pso_vshaped['fitness_history'], 'b--', linewidth=2, label='PSO (V-shaped)')
ax.plot(ga_result['fitness_history'], 'r-', linewidth=2, label='GA')
ax.axhline(BASELINE_ERR, color='gray', linestyle=':', linewidth=1.5,
           label=f'Baseline (all {N_FEATURES} features)')

ax.set_xlabel('Iteration / Generation', fontsize=12)
ax.set_ylabel('Best error rate', fontsize=12)
ax.set_title('Convergence: PSO vs GA (3-fold CV error)', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# ── Right: Feature count over time ────────────────────────────────────────────
ax2 = axes[1]

ax2.plot(pso_sigmoid['feature_count_history'], 'b-', linewidth=2, label='PSO (sigmoid)')
ax2.plot(pso_vshaped['feature_count_history'], 'b--', linewidth=2, label='PSO (V-shaped)')
ax2.plot(ga_result['feature_count_history'], 'r-', linewidth=2, label='GA')
ax2.axhline(N_FEATURES, color='gray', linestyle=':', linewidth=1.5,
             label='All features')

ax2.set_xlabel('Iteration / Generation', fontsize=12)
ax2.set_ylabel('Features in best solution', fontsize=12)
ax2.set_title('Feature Count in Best Solution', fontsize=12)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('pso_vs_ga_convergence.png', dpi=120, bbox_inches='tight')
plt.show()

# ── Summary table ──────────────────────────────────────────────────────────────
methods = ['Baseline', 'PSO (sigmoid)', 'PSO (V-shaped)', 'GA']
accuracies = [
    BASELINE_ACC,
    pso_scores.mean(),
    pso_v_scores.mean(),
    ga_scores.mean(),
]
feature_counts = [
    N_FEATURES,
    int(pso_sigmoid['best_mask'].sum()),
    int(pso_vshaped['best_mask'].sum()),
    int(ga_result['best_mask'].sum()),
]

print(f"\n{'Method':<18} {'5-fold Acc':>12} {'Features':>10} {'Reduction':>12}")
print("-" * 55)
for m, acc, nf in zip(methods, accuracies, feature_counts):
    reduction = (N_FEATURES - nf) / N_FEATURES
    print(f"{m:<18} {acc:>12.4f} {nf:>10} {reduction:>11.1%}")

## 7. PSO Sensitivity Analysis: Effect of v_max

In [ ]:
print("Testing different v_max values (20 particles, 30 iterations each)...")

v_max_values = [1.0, 2.0, 4.0, 6.0, 8.0]
v_max_results = {}

for vmax in v_max_values:
    result = binary_pso(
        n_particles=20,
        n_iterations=30,
        v_max=vmax,
        transfer_fn='sigmoid',
        record_trajectories=False,
        random_state=42,
    )
    v_max_results[vmax] = result
    print(f"  v_max={vmax:.1f}: error={result['best_fitness']:.4f}, "
          f"features={result['best_mask'].sum()}")

# Plot convergence curves
fig, ax = plt.subplots(figsize=(10, 5))
for vmax, result in v_max_results.items():
    ax.plot(result['fitness_history'], linewidth=2, label=f'v_max={vmax}')

ax.axhline(BASELINE_ERR, color='gray', linestyle=':', linewidth=1.5, label='Baseline')
ax.set_xlabel('Iteration', fontsize=12)
ax.set_ylabel('Best error rate', fontsize=12)
ax.set_title('PSO Sensitivity to Velocity Clamp (v_max)', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('pso_vmax_sensitivity.png', dpi=120, bbox_inches='tight')
plt.show()
print("\nFigure saved: pso_vmax_sensitivity.png")
print("\nNote: Very small v_max (1.0) → premature convergence (sigmoid near 0.5)")
print("Note: Very large v_max (8.0) → sigmoid saturates → stagnation")
print("Optimal: v_max = 4.0 keeps P in [0.018, 0.982]")

## 8. Key Takeaways

1. **Binary PSO** adapts continuous PSO to feature selection by converting velocity to selection probability via a transfer function. The standard choice is the S-shaped sigmoid.

2. **Velocity clamping** (`v_max = 4.0`) is essential. Without it, sigmoid probabilities reach 0 or 1 and particles stop changing — a phenomenon called velocity saturation.

3. **V-shaped transfer functions** treat positive and negative velocities symmetrically (both map to high flip probability), which can improve exploration on some landscapes.

4. **Inertia decay** (`w: 0.9 → 0.4`) transitions PSO from exploration (early, high inertia) to exploitation (late, low inertia).

5. **PSO converges faster than GA early** due to its momentum-based velocity update. GA with elitism typically matches or surpasses PSO in final solution quality given the same number of evaluations.

6. **Particle trajectories** visualise the collective decision-making process: features converge to consistently high or low selection probability as the swarm discovers the best subset.

---

**References**:
- Kennedy & Eberhart (1995). Particle swarm optimization. *Proceedings of ICNN*.
- Mirjalili & Lewis (2013). S-shaped versus V-shaped transfer functions for binary PSO. *Swarm and Evolutionary Computation*, 9, 1–14.